In [25]:
import pandas as pd
import glob

1-Importando a base de dados e juntando varias bases de dados da Leishmaniose
 

In [26]:
arquivos = glob.glob("LEIVA/*csv")

dfs = [
    pd.read_csv(arq, encoding="latin1", low_memory=False, sep=",")
    for arq in arquivos
]

df_total = pd.concat(dfs, ignore_index=True)


1.1-Mostrando a base de dados do SUS da doença Leishmaniose

In [27]:
df_total.head()

,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,DT_SIN_PRI,...,CON_INF_OU,CON_INF_BA,CON_INF_DI,CON_INF_MU,CON_INF_UF,CON_INF_PA,CON_DOENCA,CON_EVOLUC,CON_DT_OBI,CON_DT_ENC
0,2.0,B550,20150106,201501.0,2015.0,29,292080,1381.0,3016986.0,20141210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,B550,20150428,201517.0,2015.0,25,251340,1422.0,2604507.0,20140720,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2.0,B550,20150120,201503.0,2015.0,25,250840,1426.0,2606399.0,20150110,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.0,B550,20150107,201501.0,2015.0,23,230640,1515.0,2552086.0,20141107,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2.0,B550,20150707,201527.0,2015.0,23,230523,5596.0,2481553.0,20150707,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
manter = ['DT_OBITO', 'EVOLUCAO', 'CS_GESTANT']

df_limpo = df_total.loc[
    :, (df_total.isna().mean() <= 0.15) | (df_total.columns.isin(manter))
]


In [29]:
df_limpo.head()



,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_UNIDADE,DT_SIN_PRI,SEM_PRI,ANO_NASC,CS_SEXO,...,FRAQUEZA,EMAGRA,TOSSE,BACO,FIGADO,HIV,IFI,OUTRO,EVOLUCAO,DT_OBITO
0,20150106,201501.0,2015.0,29,292080,3016986.0,20141210,201450.0,1996.0,M,...,1.0,1.0,1.0,2.0,2.0,9.0,1.0,3.0,1.0,
1,20150428,201517.0,2015.0,25,251340,2604507.0,20140720,201430.0,1979.0,F,...,1.0,2.0,1.0,1.0,1.0,9.0,2.0,2.0,1.0,
2,20150120,201503.0,2015.0,25,250840,2606399.0,20150110,201501.0,1958.0,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
3,20150107,201501.0,2015.0,23,230640,2552086.0,20141107,201445.0,2009.0,M,...,1.0,1.0,2.0,1.0,1.0,2.0,1.0,3.0,1.0,
4,20150707,201527.0,2015.0,23,230523,2481553.0,20150707,201527.0,NaN,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,


In [30]:
df_limpo = df_limpo.drop(columns=['EMAGRA', 'TOSSE', 'BACO', 'FIGADO','IFI','OUTRO','ID_PAIS','FEBRE','FRAQUEZA','DT_SIN_PRI','SEM_PRI', 'CS_GESTANT','CS_RACA','HIV','EVOLUCAO','ID_MN_RESI','SEM_NOT'])


In [31]:
df_limpo.head()

,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_UNIDADE,ANO_NASC,CS_SEXO,SG_UF,DT_OBITO
0,20150106,2015.0,29,292080,3016986.0,1996.0,M,29.0,
1,20150428,2015.0,25,251340,2604507.0,1979.0,F,25.0,
2,20150120,2015.0,25,250840,2606399.0,1958.0,M,25.0,
3,20150107,2015.0,23,230640,2552086.0,2009.0,M,23.0,
4,20150707,2015.0,23,230523,2481553.0,NaN,M,23.0,


In [32]:
df_limpo['DT_OBITO'].value_counts()

DT_OBITO
            156504
20170508         9
20221221         8
20240610         8
20150226         7
             ...  
20070116         1
20070401         1
20070508         1
20070512         1
20110108         1
Name: count, Length: 5018, dtype: int64

2-Descobrindo a linha onde começa o cabeçalho da base de dados?

In [33]:
with open('CODIGO_IBGE.csv', encoding='latin1') as f:
    for i, linha in enumerate(f):
        if linha.startswith('UF,'):
            print(i, linha)
            break



6 UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município,Distrito,Código de Distrito Completo,Nome_Distrito,OBS,,Código de Distrito Completo,Nome_Distrito,,,,,,,,,



3-Descobrindo q é numero 6 onde começa o cabeçalho, fazendo a leitura e o tratamento dos dados

In [34]:
df_ibge = pd.read_csv(
    'CODIGO_IBGE.csv',
    sep=',',
    encoding='latin1',
    skiprows=6,
    header=0,
    engine='python'
)



4-Mostrado as linhas e as colunas

In [35]:
df_ibge.head()
df_ibge.columns


Index(['UF', 'Nome_UF', 'Região Geográfica Intermediária',
       'Nome Região Geográfica Intermediária', 'Região Geográfica Imediata',
       'Nome Região Geográfica Imediata', 'Município',
       'Código Município Completo', 'Nome_Município', 'Distrito',
       'Código de Distrito Completo', 'Nome_Distrito', 'OBS', 'Unnamed: 13',
       'Código de Distrito Completo.1', 'Nome_Distrito.1', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24'],
      dtype='object')

5-Removendo todas as colunas Unnamed.

In [36]:
df_ibge = df_ibge.loc[:, ~df_ibge.columns.str.startswith('Unnamed')]


6-Mostrando a base de dados de codigo Municipal do IBGE

In [37]:
df_ibge.head()

,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município,Distrito,Código de Distrito Completo,Nome_Distrito,OBS,Código de Distrito Completo.1,Nome_Distrito.1
0,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,5,110001505,Alta Floresta D'Oeste,NaN,110001505.0,Alta Floresta D'Oeste
1,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,15,110001515,Filadélfia d'Oeste,NaN,110001515.0,Filadélfia d'Oeste
2,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,20,110001520,Izidolândia,NaN,110001520.0,Izidolândia
3,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,25,110001525,Nova Gease d'Oeste,NaN,110001525.0,Nova Gease d'Oeste
4,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,30,110001530,Rolim de Moura do Guaporé,NaN,110001530.0,Rolim de Moura do Guaporé


In [38]:
df_ibge = df_ibge.drop(columns=['Município', 'Distrito', 'Código de Distrito Completo','Nome_Distrito', 'OBS', 'Código de Distrito Completo.1', 'Nome_Distrito.1'])

In [39]:
df_ibge.head()

,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Código Município Completo,Nome_Município
0,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste
1,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste
2,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste
3,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste
4,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste


In [40]:
df_ibge.to_csv('dados_tratados_ibge.csv', index=False)


In [41]:
sus = pd.read_csv("dados_sus_tratados.csv", dtype=str)
ibge = pd.read_csv("dados_tratados_ibge.csv", dtype=str)

In [42]:
sus['ID_MUNICIP'] = sus['ID_MUNICIP'].str.zfill(7)
ibge['Código Município Completo'] = ibge['Código Município Completo'].str.zfill(7)


In [43]:
sus_merge = sus.merge(
    ibge[['Código Município Completo']],
    left_on='ID_MUNICIP',
    right_on='Código Município Completo',
    how='left'
)


In [44]:
sus.head()

,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_UNIDADE,ANO_NASC,CS_SEXO,SG_UF,DT_OBITO
0,20150106,2015.0,29,0292080,3016986.0,1996.0,M,29.0,
1,20150428,2015.0,25,0251340,2604507.0,1979.0,F,25.0,
2,20150120,2015.0,25,0250840,2606399.0,1958.0,M,25.0,
3,20150107,2015.0,23,0230640,2552086.0,2009.0,M,23.0,
4,20150707,2015.0,23,0230523,2481553.0,NaN,M,23.0,


In [45]:
cols = ['NU_ANO', 'ID_UNIDADE', 'ANO_NASC', 'SG_UF']

sus[cols] = sus[cols].apply(
    lambda c: pd.to_numeric(c, errors='coerce')
)

In [46]:
sus[cols] = sus[cols].astype('Int64')

In [47]:
sus.head()


,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_UNIDADE,ANO_NASC,CS_SEXO,SG_UF,DT_OBITO
0,20150106,2015,29,0292080,3016986,1996,M,29,
1,20150428,2015,25,0251340,2604507,1979,F,25,
2,20150120,2015,25,0250840,2606399,1958,M,25,
3,20150107,2015,23,0230640,2552086,2009,M,23,
4,20150707,2015,23,0230523,2481553,<NA>,M,23,
